# Genetic Algorithm Meal Planner

In [1]:
from models import (
    Pantry,
    Ingredient,
    PantryIngredient,
    NutritionalInformation,
    MealPlanningEnvironment,
    UserPreferences,
)
from datetime import datetime, timedelta
from GAMealPlan import evaluate_meal_plan
from pygad import GA
import json
from random import randint, seed, random

In [2]:
preferences = UserPreferences(
    weekly_budget=50,
    calorie_target_per_day=2500,
    protein_target_per_day=50,
    is_vegetarian=False,
    is_vegan=False,
    requires_gluten_free=False,
    requires_lactose_free=False,
)

In [3]:
all_ingredients = []

with open("./recipe_extraction/supplemented_structured_ingredients.json", "r") as f:
    ingredients_data = json.load(f)

    for ingredient_data in ingredients_data:
        ingredient_nutritional_information = NutritionalInformation(**ingredient_data["nutritional_information"])

        ingredient = Ingredient(
            name=ingredient_data["name"], nutritional_information=ingredient_nutritional_information
        )
        all_ingredients.append(ingredient)

In [4]:
def get_ingredient(ingredient_name: str) -> Ingredient | None:
    return next((ingredient for ingredient in all_ingredients if ingredient.name == ingredient_name), None)

In [5]:
seed(1)

In [6]:
def get_pantry_ingredient(
    ingredient_name: str, estimated_expiration_date: datetime, estimated_financial_cost: float
) -> PantryIngredient:
    ingredient = get_ingredient(ingredient_name)

    if ingredient is None:
        raise ValueError(f"Ingredient '{ingredient_name}' not found in ingredient list")

    return PantryIngredient(
        name=ingredient.name,
        nutritional_information=ingredient.nutritional_information,
        estimated_expiration_date=estimated_expiration_date,
        estimated_financial_cost=estimated_financial_cost,
    )

In [7]:
CURRENT_DATE = datetime.now()

In [8]:
pantry_ingredient_name_by_quantity = {
    "chicken breast": 200,
    "broccoli": 150,
    "rice": 100,
}

In [9]:
pantry_ingredients = [
    get_pantry_ingredient(ingredient_name, CURRENT_DATE + timedelta(days=randint(1, 7)), random())
    for ingredient_name, quantity in pantry_ingredient_name_by_quantity.items()
]

In [10]:
pantry_ingredient_by_quantity = dict(zip(pantry_ingredients, pantry_ingredient_name_by_quantity.values()))

In [11]:
pantry = Pantry()

for pantry_ingredient, quantity in pantry_ingredient_by_quantity.items():
    pantry.add(
        pantry_ingredient,
        quantity,
    )

In [12]:
meal_planning_environment = MealPlanningEnvironment(
    recipes=[],
    pantry=pantry,
    preferences=preferences,
)

In [13]:
meal_planning_environment.load_recipes_from_json("./recipe_extraction/supplemented_structured_recipes.json")

In [14]:
recipes = meal_planning_environment.recipes
pantry_stock = meal_planning_environment.pantry.stock
days_until_expiry = meal_planning_environment.pantry.get_days_until_expiry(CURRENT_DATE)
ingredient_costs = meal_planning_environment.pantry.ingredient_costs
preferences = meal_planning_environment.preferences

In [21]:
def fitness_function(_ga_instance, solution, _solution_index):
    return evaluate_meal_plan(solution, recipes, pantry_stock, days_until_expiry, preferences, ingredient_costs)

In [24]:
GENERATION_PRINT_INTERVAL = 10

In [25]:
def on_generation(ga_instance):
    num_generations = ga_instance.generations_completed

    if num_generations % GENERATION_PRINT_INTERVAL == 0:
        best_fitness = ga_instance.best_solution()[1]
        print(f"Generation {num_generations}, Best Fitness: {best_fitness:.2f}")

In [18]:
NUM_GENERATIONS = 1000
NUM_PARENTS_MATING = 20
POPULATION_SIZE = 100
NUM_GENES = 21  # 3 meals/day x 7 days

In [26]:
ga_instance = GA(
    num_generations=NUM_GENERATIONS,
    num_parents_mating=NUM_PARENTS_MATING,
    fitness_func=fitness_function,
    sol_per_pop=POPULATION_SIZE,
    num_genes=NUM_GENES,
    gene_type=int,
    gene_space=list(range(len(meal_planning_environment.recipes))),
    parent_selection_type="tournament",
    K_tournament=5,
    keep_elitism=1,
    crossover_type="single_point",
    crossover_probability=0.8,
    mutation_type="random",
    mutation_probability=0.1,
    on_generation=on_generation,
    random_seed=1,
)

In [27]:
ga_instance.run()

Generation 10, Best Fitness: -55.44
Generation 20, Best Fitness: -24.04
Generation 30, Best Fitness: -22.50
Generation 40, Best Fitness: -19.60
Generation 50, Best Fitness: -17.31
Generation 60, Best Fitness: -17.31
Generation 70, Best Fitness: -15.35
Generation 80, Best Fitness: -13.73
Generation 90, Best Fitness: -13.73
Generation 100, Best Fitness: -12.42
Generation 110, Best Fitness: -11.78
Generation 120, Best Fitness: -11.78
Generation 130, Best Fitness: -10.63
Generation 140, Best Fitness: -9.81
Generation 150, Best Fitness: -8.62
Generation 160, Best Fitness: -8.62
Generation 170, Best Fitness: -8.62
Generation 180, Best Fitness: -8.23
Generation 190, Best Fitness: -8.23
Generation 200, Best Fitness: -8.23
Generation 210, Best Fitness: -8.23
Generation 220, Best Fitness: -7.63
Generation 230, Best Fitness: -7.06
Generation 240, Best Fitness: -6.97
Generation 250, Best Fitness: -6.97
Generation 260, Best Fitness: -6.97
Generation 270, Best Fitness: -6.97
Generation 280, Best Fit

In [28]:
best_meal_plan, best_fitness, _ = ga_instance.best_solution()

print(f"Best fitness score: {best_fitness:.2f}")

Best fitness score: 4.15


In [29]:
for day in range(7):
    print(f"\nDay {day + 1} meal plan:")

    for meal in range(3):
        recipe = meal_planning_environment.recipes[int(best_meal_plan[day * 3 + meal])]

        print(f"Meal {meal + 1}: {recipe.name}")


Day 1 meal plan:
Meal 1: Diner-Style Bacon for a Crowd
Meal 2: Turkey and Zucchini Meat Loaf
Meal 3: Peanut Butter Pie

Day 2 meal plan:
Meal 1: Vegetable-Stuffed Loin of Veal with Sweetbreads
Meal 2: Ras-El-Hanout
Meal 3: Roast Turkey with Pesto-Rice Stuffing

Day 3 meal plan:
Meal 1: Chicken with Olives and Feta Cheese
Meal 2: Roasted Broccoli Florets with Toasted Breadcrumb Gremolata
Meal 3: "Like a Caesar"

Day 4 meal plan:
Meal 1: Aunt Tom's Italian Cream Cake
Meal 2: Clam and Bacon Pizza
Meal 3: Zucchini Salad With Ajo Blanco Dressing & Spiced Nuts

Day 5 meal plan:
Meal 1: Creamy Horseradish Potato Salad
Meal 2: Veal Scallops with Bacon and Potatoes
Meal 3: Tart Shell

Day 6 meal plan:
Meal 1: Toasted Nori Mayonnaise
Meal 2: "Paella" Fried Rice
Meal 3: Celery and Fennel with Bacon

Day 7 meal plan:
Meal 1: Nancy's Falafel Pistachios
Meal 2: Rib-Eye Steaks with Roasted Red Peppers and Balsamic Vinegar
Meal 3: Sunflower Seed "Risotto" With Squash and Mushrooms


In [30]:
# going through the best meal plan and calculating how much of each ingredient is consumed from the pantry

consumed_stock = dict.fromkeys(pantry_stock.keys(), 0)

for index in best_meal_plan:
    recipe = recipes[int(index)]

    for ingredient_name, quantity_needed in recipe.ingredients.items():
        available = pantry_stock.get(ingredient_name, 0) - consumed_stock.get(ingredient_name, 0)
        from_pantry = max(0, min(available, quantity_needed))
        consumed_stock[ingredient_name] = consumed_stock.get(ingredient_name, 0) + from_pantry


In [31]:
import pandas as pd

pantry_data = []

for ingredient in pantry.ingredients:
    available = pantry_stock[ingredient.name]
    used = consumed_stock.get(ingredient.name, 0)
    unused = max(0, available - used)
    days = days_until_expiry[ingredient.name]
    expiry_str = str(days) + "d"
    pantry_data.append(
        {
            "Ingredient": ingredient.name,
            "Available": available,
            "Consumed": used,
            "Unused": unused,
            "Expires in": expiry_str,
        }
    )

pantry_data_df = pd.DataFrame(pantry_data)

pantry_data_df

,Ingredient,Available,Consumed,Unused,Expires in
0,chicken breast,200,0.0,200,2d
1,broccoli,150,150.0,0,7d
2,rice,100,100.0,0,3d


In [32]:
meal_plan = []

for day in range(7):
    meal_indices = best_meal_plan[day * 3 : day * 3 + 3]
    meal_names = [recipes[int(index)].name for index in meal_indices]
    meal_plan.append(
        {
            "Day": day + 1,
            "Meal 1": meal_names[0],
            "Meal 2": meal_names[1],
            "Meal 3": meal_names[2],
        }
    )

meal_plan_df = pd.DataFrame(meal_plan)

meal_plan_df

,Day,Meal 1,Meal 2,Meal 3
0,1,Diner-Style Bacon for a Crowd,Turkey and Zucchini Meat Loaf,Peanut Butter Pie
1,2,Vegetable-Stuffed Loin of Veal with Sweetbreads,Ras-El-Hanout,Roast Turkey with Pesto-Rice Stuffing
2,3,Chicken with Olives and Feta Cheese,Roasted Broccoli Florets with Toasted Breadcru...,"""Like a Caesar"""
3,4,Aunt Tom's Italian Cream Cake,Clam and Bacon Pizza,Zucchini Salad With Ajo Blanco Dressing & Spic...
4,5,Creamy Horseradish Potato Salad,Veal Scallops with Bacon and Potatoes,Tart Shell
5,6,Toasted Nori Mayonnaise,"""Paella"" Fried Rice",Celery and Fennel with Bacon
6,7,Nancy's Falafel Pistachios,Rib-Eye Steaks with Roasted Red Peppers and Ba...,"Sunflower Seed ""Risotto"" With Squash and Mushr..."


In [33]:
shopping_list = {}

for index in best_meal_plan:
    recipe = recipes[int(index)]

    for ingredient_name, quantity_needed in recipe.ingredients.items():
        available = pantry_stock.get(ingredient_name, 0) - consumed_stock.get(ingredient_name, 0)
        to_buy = max(0, quantity_needed - available)
        if to_buy > 0:
            shopping_list[ingredient_name] = shopping_list.get(ingredient_name, 0.0) + to_buy

shopping_data = [
    {
        "Ingredient": name,
        "Quantity to Buy (g)": round(qty, 1),
        "Cost (€)": round((qty / 100.0) * ingredient_costs.get(name, 1.0), 2),
    }
    for name, qty in sorted(shopping_list.items())
]

shopping_df = pd.DataFrame(shopping_data)

total_cost = shopping_df["Cost (€)"].sum()
total_row = pd.DataFrame([{"Ingredient": "TOTAL", "Quantity to Buy (g)": "", "Cost (€)": round(total_cost, 2)}])
shopping_df = pd.concat([shopping_df, total_row], ignore_index=True)

print(f"Shopping list ({len(shopping_data)} ingredients, total cost: €{total_cost:.2f})\n")
shopping_df


Shopping list (151 ingredients, total cost: €51.00)



,Ingredient,Quantity to Buy (g),Cost (€)
0,"#1$2"" cubed butternut squash",78.9,0.79
1,(green) pumpkin seeds,6.7,0.07
2,Bacon slices,3.7,0.04
3,Cool Whip,75.6,0.76
4,Dijon mustard,2.0,0.02
...,...,...,...
147,white bread,14.0,0.14
148,white-wine vinegar,1.8,0.02
149,zucchini,13.6,0.14
150,zucchinis,8.0,0.08


In [34]:
nutrition_data = []

for day in range(7):
    meal_indices = best_meal_plan[day * 3 : day * 3 + 3]
    daily_calories = sum(recipes[int(i)].nutritional_information.calories or 0.0 for i in meal_indices)
    daily_protein = sum(recipes[int(i)].nutritional_information.protein or 0.0 for i in meal_indices)
    nutrition_data.append(
        {
            "Calories (kcal)": round(daily_calories, 1),
            "Δ Calories and Target Calories (kcal)": round(daily_calories - preferences.calorie_target_per_day, 1),
            "Protein (g)": round(daily_protein, 1),
            "Δ Protein and Target Protein (g)": round(daily_protein - preferences.protein_target_per_day, 1),
        }
    )

nutrition_df = pd.DataFrame(nutrition_data, index=["Day 1", "Day 2", "Day 3", "Day 4", "Day 5", "Day 6", "Day 7"])

nutrition_df


,Calories (kcal),Δ Calories and Target Calories (kcal),Protein (g),Δ Protein and Target Protein (g)
Day 1,2508.7,8.7,51.0,1.0
Day 2,2486.7,-13.3,63.0,13.0
Day 3,2536.2,36.2,50.7,0.7
Day 4,2489.4,-10.6,50.7,0.7
Day 5,2508.9,8.9,55.0,5.0
Day 6,2486.7,-13.3,55.5,5.5
Day 7,2528.0,28.0,48.9,-1.1
